In [1]:
import os
import os.path
import sys
import numpy as np
import networkx as nx
import itertools as it
import random as rd
import math
import pickle as pk
import pandas as pd
from collections import defaultdict, Counter, ChainMap
from scipy.spatial import distance_matrix
import matplotlib.pyplot as plt
# %matplotlib inline

plt.rcParams['font.family'] = 'Arial'

**huMAP3.0 complexes**

- PI3K
- CREB
- Phosphatases

In [3]:
p = '../../data/processed/cytoscape/selected_humap3.0_complexes.tsv'
complexes = pd.read_csv(p, delimiter='\t')

active_baits = [
    'BrafG469A', 'ArafS214P', 'HrasQ61R', 'HrasG13R', 
    'NrasQ61R', 'NrasQ61K', 
    'KrasG12V', 'KrasG12D',
    'ERK2DD', 'ERK1DD', 'MEK2DD', 'MEK1DD', 
    'Raf1S259F', 'Raf1S257L', 'BrafV600E']

inactive_baits = [
    'BrafD594N',  'ArafD447A', 
    'HrasS17N', 'NrasS17N', 'KrasS17N', 
    'ERK2AA', 'ERK1AA', 'MEK2AA', 'MEK1AA',
    'Raf1D486A']

wt_baits =  [ "BrafWT", "ArafWT", "HrasWT", "NrasWT", 
             "KrasWT","ERK2WT", "ERK1WT", "MEK2WT", 
             "MEK1WT", "Raf1WT"]

prot2col = {
    'KRAS': '#f26924',
    'NRAS': '#f68b22',
    'HRAS': '#faa61e',
    'ARAF': '#174a92',
    'BRAF': '#4f88c7',
    'RAF1': '#a1bce2',
    'MEK1': '#009147',
    'MEK2': '#88be40',
    'ERK1': '#a62877',
    'ERK2': '#dfa0c2'
}


active_baits = active_baits[::-1]
inactive_baits = inactive_baits[::-1]
wt_baits = wt_baits[::-1]

active_baits = [x.upper() for x in active_baits]
inactive_baits = [x.upper() for x in inactive_baits]
wt_baits = [x.upper() for x in wt_baits]

In [4]:
import networkx as nx
import matplotlib.pyplot as plt
import random

# --- Groups ---
def build_graph(humap3_complex: pd.DataFrame, state: str):
    group1 = wt_baits
    
    if state == 'Active':
        group1 = active_baits
    elif state == 'WT':
        group1 = wt_baits
    else:
        group1 = inactive_baits
        
    
    
    colors = []
    for bait in group1:
        for prot, col in prot2col.items():
            if prot in bait:
                colors.append(col)
    
    # Build network
    # add baits
    G = nx.Graph()
    for bait, col in zip(group1, colors):
        G.add_node(bait, bait="yes", color=col)

    group2 = set(humap3_complex['to'].values)
    G.add_nodes_from(group2, bait="no", color='#cccccc')

    df = humap3_complex[humap3_complex['from'].isin(group1)]
    for _, row in df.iterrows():
        G.add_edge(row["from"], row["to"], interaction=row['strict_type'])

    # add prey-prey edges
    df = humap3_complex[~humap3_complex['from'].isin(group1)]
    for _, row in df.iterrows():
        G.add_edge(row["from"], row["to"], interaction=row['strict_type'])

    # layout baits in circle, preys inside circle
    pos_group1 = nx.circular_layout(group1, scale=3)
    pos = nx.spring_layout(G, pos=pos_group1, fixed=group1, k=0.7, scale=2, seed=14)
    return G, pos_group1, pos, group1, group2

def move2center(pos, group2):
    x = [] 
    y = []
    for elem in group2:
        a = pos[elem]
        x.append(a[0])
        y.append(a[1])

    for elem in group2:
        a = pos[elem]
        pos[elem] = [ a[0] - (np.mean(x) ), a[1] - (np.mean(y)) ]
    return pos


def draw_network(G, group1, group2, pos, ax, title=None):
    pos = move2center(pos, group2)
    plt.figure(figsize=(7, 7))
    # Draw nodes
    nx.draw_networkx_nodes(G, pos, nodelist=group1, node_color="#bdbdbd",
                           node_shape="D", node_size=400, edgecolors="none", ax=ax )
    nx.draw_networkx_nodes(G, pos, nodelist=group2, node_color="#bdbdbd",
                           node_shape="o", node_size=200, edgecolors="none", ax=ax)

    # Split edges by type
    edges_pp = [(u, v) for u, v, d in G.edges(data=True) if d["interaction"] == "both"]
    edges_apms = [(u, v) for u, v, d in G.edges(data=True) if d["interaction"] == "AP-MS"]
    edges_tid = [(u, v) for u, v, d in G.edges(data=True) if d["interaction"] == "TbID"]
    edges_cp = [(u, v) for u, v, d in G.edges(data=True) if d["interaction"] == "CORUM"]
    edges_sg = [(u, v) for u, v, d in G.edges(data=True) if d["interaction"] == "SIGNOR"]

    # Draw edges
    nx.draw_networkx_edges(G, pos, edgelist=edges_pp, edge_color="grey", width=1.5, alpha=0.8,ax=ax)
    nx.draw_networkx_edges(G, pos, edgelist=edges_apms, edge_color="#fb9a99", width=1.5, alpha=0.8,ax=ax)
    nx.draw_networkx_edges(G, pos, edgelist=edges_tid, edge_color="#a6cee3", width=1.5, alpha=0.8,ax=ax)
    nx.draw_networkx_edges(G, pos, edgelist=edges_cp, edge_color="black", width=2.5, alpha=0.8,ax=ax)
    nx.draw_networkx_edges(G, pos, edgelist=edges_sg, edge_color="#33a02c", width=2.5, alpha=0.8,ax=ax)

    label_pos = {node: (x, y - 0.3) for node, (x, y) in pos.items()}  # shift downward
    nx.draw_networkx_labels(G, label_pos, font_size=12, ax=ax)

def relabel_and_shift_graph(G, pos, prefix, shift=(0, 0), scale=55):
    """
    Relabel nodes with a prefix, scale + shift positions,
    return relabeled graph, position dict, and mapping.
    """
    mapping = {node: f"{prefix}_{node}" for node in G.nodes()}
    new_G = nx.relabel_nodes(G, mapping)
    
    # rescale + shift positions
    new_pos = {
        mapping[node]: (coord[0] * scale + shift[0], coord[1] * scale + shift[1])
        for node, coord in pos.items()
    }
    
    # keep original label as attribute
    nx.set_node_attributes(new_G, {mapping[node]: node for node in G.nodes()}, name="orig_label")
    
    return new_G, new_pos, mapping

In [5]:
base_dir = '../../data/processed/cytoscape/graphml/'
for elem in set(complexes['category'] ):
    category = complexes[complexes['category'] == elem ]
    G_inactive, pos_group1, pos_inactive, group1, group2 = build_graph(category[category['network'] == 'Inactive'], 'Inactive')
    #
    G_wt, pos_group1, pos_wt, group1, group2 = build_graph(category[category['network'] == 'WT'], 'WT')
    #
    G_active, pos_group1, pos_active, group1, group2 = build_graph(category[category['network'] == 'Active'], 'Active')
    ##
    ##
    # Example: assume you have G1, G2, G3 with pos1, pos2, pos3 already computed
    shift1 = (-600, 0)  # left
    shift2 = (0, 0)      # center
    shift3 = (600, 0)   # right

    G1_pref, pos1_pref, map1 = relabel_and_shift_graph(G_inactive, pos_inactive, "Inactive", shift1)
    G2_pref, pos2_pref, map2 = relabel_and_shift_graph(G_wt, pos_wt, "WT", shift2)
    G3_pref, pos3_pref, map3 = relabel_and_shift_graph(G_active, pos_active, "Active", shift3)

    # Merge graphs (union without duplicate edges)
    G_merged = nx.compose_all([G1_pref, G2_pref, G3_pref])

    # Merge positions
    pos_merged = {}
    pos_merged.update(pos1_pref)
    pos_merged.update(pos2_pref)
    pos_merged.update(pos3_pref)

    # Also save positions for Cytoscape as node attributes
    nx.set_node_attributes(G_merged, {n: float(x) for n, (x, y) in pos_merged.items()}, "x")
    nx.set_node_attributes(G_merged, {n: float(y) for n, (x, y) in pos_merged.items()}, "y")
    nx.write_graphml(G_merged, base_dir + elem + ".graphml")